# Task 20 — Combining Data, Cricket Records, and the Iris Dataset

This notebook is the grand finale of the pandas progression: it demands every skill you have built — concatenation, merging, grouping, cleaning, masking, and charting — and it throws them all at three completely different datasets. Do not read it as one task; read it as three mini-projects bolted into a single file.

Project one is a tiny online-store database. The four files — items, quarter-1, quarter-2, and quarter-3 — split an order stream across three months with a separate item catalogue. Q-1 to Q-5 first stack the three quarter files into one complete table with a Q-1 / Q-2 / Q-3 index, then answer per-quarter business questions: which items sold most, which items earned the most revenue, and what the average order price was — with the price column converted from its raw '$7.75'-style text into real rupees first. This part practises pd.concat with keys, plus per-quarter groupby aggregations.

Project two is the IPL ball-by-ball log. Q-6 computes each season's purple-cap holder — the most wickets in a season, with the bowler's economy as the tiebreaker. Q-7 repeats the logic for the death overs, overs 16-20 only. Q-8 builds a function that returns one complete season-by-season record for any batsman — Innings, TotalRuns, Average, HighestScore, StrikeRate. Q-9 merges the match summaries with the ball-by-ball data to build a player-of-the-match report of the batting figure (runs/balls) and bowling figure (wickets/runs conceded) for every game. These are the hardest problems in the notebook because the masks (over 16+, player_out, extra_type, wide versus no-ball) must all be right before any number can be trusted.

Project three is the classic Iris dataset, split into five fragments — all sepal measurements, all petal measurements, and three species files. The sheet's Q-9 (numbered 9 a second time) plots the average sepal length for Virginica against the average petal length for Setosa. Q-10 reassembles the fragments into one complete, correctly-ordered six-column dataset and shuffles it with `sample()`. Q-11 groups the merged frame by Species and reports the minimum and maximum sepal and petal widths for Setosa and Versicolor.

Work the three projects in order and verify every answer against the sample outputs. Read, merge, group, filter — the whole session is that chain, repeated at higher and higher stakes.

In [ ]:
import pandas as pd
import numpy as np

# 1. The Quarter-Revenue Datasets

## 1.1 Four files, one business

The opening cells load the store's raw ingredients:

```python
items = pd.read_csv(items_path)
q1 = pd.read_csv(q1_path)
q2 = pd.read_csv(q2_path)
q3 = pd.read_csv(q3_path)
```

`items.csv` is the catalogue: `item_id` and `item_name`. Each `quarter-*.csv` is one month of orders with the columns `['order_id', 'quantity', 'item_id', 'choice_description_id', 'item_price']`. `item_id` is the shared key that connects the catalogue to the orders — the future join column. The paths point to Google-Drive-mounted files (`/content/drive/MyDrive/...`), staged for this task.

Two traps are planted before the questions even start. First, duplicate `order_id` values: an order with several different items appears in several rows with the same order_id, so 'average order price' is NOT `item_price.mean()` — it is the mean of the order totals (each order's price summed over its rows). Second, `item_price` is stored as TEXT with a dollar sign, like `'$7.75'`. The question note says so explicitly: convert `item_price` into rupees first, using the same pipeline you practised — strip the `$`, convert to numeric, and scale by the rupee rate you used in the earlier task.

> Hint: peek at each quarter with `q1.head()` before combining; confirm the price column really is a string with `item_price.dtype`.

In [ ]:
# import like this
items_path = "/content/drive/MyDrive/DSMP/01 - Python/Daily Tasks/Week 7/data/items.csv"
q1_path = "/content/drive/MyDrive/DSMP/01 - Python/Daily Tasks/Week 7/data/quarter-1.csv"
q2_path = "/content/drive/MyDrive/DSMP/01 - Python/Daily Tasks/Week 7/data/quarter-2.csv"
q3_path = "/content/drive/MyDrive/DSMP/01 - Python/Daily Tasks/Week 7/data/quarter-3.csv"


q1= pd.read_csv(q1_path)
q2 = pd.read_csv(q2_path)
q3 = pd.read_csv(q3_path)

items = pd.read_csv(items_path)

# 2. Q-1 to Q-5 — Combining the Quarters

### Q:1-5
1. You are given three quarter files; your job is to append these three files and make a single dataframe.
2. Have an index as Q-1, Q-2, Q-3 for the respective quarter files in the dataframe.
3. You are given a file `items.csv` which has `item_id` and `item_name`. Find out the most sold items in each quarter.
4. Find out items which have made the most revenue in each quarter.
5. Find out the average order price of each quarter.

Steps 1 and 2 fall to one call: `pd.concat([q1, q2, q3], keys=['Q-1', 'Q-2', 'Q-3'])`. The `keys=` argument labels every block of rows with its quarter, so the combined frame carries a top MultiIndex level of Q-1 / Q-2 / Q-3 — exactly the index the question asks for. Deliberately do NOT use `ignore_index=True` here; it would drop the keys.

Step 3 — most sold per quarter — is a groupby over quantity: group by the quarter level and sum `quantity`, then take the highest per quarter. Better, join the item names first so the answer reads 'which ITEM' rather than which id: merge the combined frame with items on `item_id`, then groupby ['quarter', 'item_name'].

Step 4 — most revenue per quarter — first creates a revenue column: `revenue = quantity * rupees_price`. Then group by quarter and item, sum the revenue, and take the top per quarter. Steps 3 and 4 are the same shape; one sums quantities, the other multiplies before summing.

Step 5 — average order price — respects the multi-row order structure: total each order's price by `groupby('order_id')['revenue'].sum()`, then average over orders per quarter. The output is one average price per quarter; a quarter whose orders carried more expensive items shows a higher average.

> Hint: concat keys -> merge items -> multiply quantity * price -> groupby quarter (+ item) -> sum or mean.

In [ ]:
# code here

# 3. Q-6 — The Purple Cap per Season

### Q-6: From the IPL dataset find out the Purple cap holder each season.

> Note: the bowler with the most number of wickets in a season gets the Purple Cap. If more than one bowler has the same number of wickets in the season, the one with the least economy among them is the Purple Cap holder.
> Bowler's Economy = runs conceded per six balls.

A bowler's wicket count comes from the rows where they are recorded as the dismisser. In the ball-by-ball frame, a delivery has the `bowler` column, and `player_out` names the dismissed batsman when a wicket fell — so count per (season, bowler) the rows where `player_out` is non-null.

Economy is 'runs conceded per six balls': total runs conceded divided by balls bowled, times six — `(runs_conceded / balls_bowled) * 6`. Sum `batsman_runs` plus extras across the bowler's deliveries for the conceded total, and count the deliveries for balls. The groupby per season is two-level: `ipl.groupby(['season', 'bowler'])` with a dictionary agg — wickets from player_out, runs_conceded from batsman_runs, balls from a count — then compute the economy inside or after the groupby.

Selection per season: sort by wickets descending, then economy ascending (least runs per six balls), and take the first bowler of each season — the same drop_duplicates(subset='season', keep='first') trick you used for the orange cap. The output is one Purple Cap holder per season, from 2008 onward.

> Hint: wickets = player_out notnull count per (season, bowler); economy = runs / balls * 6; sort wickets desc then economy asc, keep first per season.

In [ ]:
# code here

# 4. Q-7 — The Best Bowler in Death Overs

### Q-7: Find the best bowler in death overs.
> Note: the bowler with the most number of wickets; in case of a tie, the least economy is better.
> Death Overs - [16-20].

Death overs are the last five overs of an innings — overs 16 to 20 — where games are won and lost. The single most important line in this question is the filter: `ipl[ipl['over'] >= 16]` keeps exactly those death-over deliveries. Check the max value of the `over` column first: if the frame stores overs 1..20, use `>= 16`; if it stores 0..19, use `>= 15`.

Once the frame is cut down, the pipeline is identical to Q-6: per bowler, count wickets (non-null player_out), sum runs conceded, and count deliveries. Sort by wickets descending and, as the tiebreak, economy ascending. The winner is the death-overs specialist who strikes most often while staying economical — a genuine cricket answer, not just a sort: the bowler at the top has combined the two disciplines.

This question is deliberately Q-6 with one extra mask. If the Purple Cap pipeline works, the death-over answer is one added line.

> Hint: mask overs >= 16 first; then the same groupby, sort by wickets desc and economy asc, head(1).

In [ ]:
# code here

# 5. Q-8 — Season-by-Season Batsman Record

### Q-8: Make a function which takes a `batsman_name` input and returns a dataframe. Columns of the dataframe are `['Season', 'Innings', 'TotalRuns', 'Avg', 'HighestScore', 'StrikeRate']`. Make the `Season` column the index.

* Avg - total_runs / number of times got out - the `player_out` column will help.
* StrikeRate - (total_runs / balls faced) * 100 - wides are NOT included in the batsman's ball faced counts. No balls ARE included. The `extra_type` column will help.
* Batsman can score runs on No Balls.
* Batsman can get out on No Ball or Wides. And even while being on the non-striker's end. Keep these things in mind before masking.

A function that returns a whole dataframe per batsman, built from careful masks. Its skeleton:

```python
def batsman_season_record(batsman_name):
    ...
    return record_df   # Season as index, columns as specified
```

Every aggregate is over the batsman's rows, grouped by season. Work the five columns:

Innings: count the distinct matches where the batsman batted — `groupby('season')['match_id'].nunique()`.

TotalRuns: sum `batsman_runs` per season.

Avg: `TotalRuns / dismissals`, and dismissals must include every way of getting out — the `player_out` column already names the batsman dismissed, but the note reminds you a batsman can be out on no-balls and wides, and even as a non-striker run out. So do NOT filter out those balls before counting `player_out == batsman`.

StrikeRate: `(runs / balls_faced) * 100`. The note is specific about which balls count: wides are NOT balls faced; no-balls ARE. Filter the batsman's deliveries by `extra_type`: exclude 'wides', include 'no balls' along with normal deliveries.

Build the dataframe from the per-season aggregates, set Season as the index, and return it. The function should greet any name in the file — it works for any batsman you pass.

> Hint: groupby('season') with nunique for innings, sum for runs, careful player_out non-null counts for Avg, and a wide-vs-no-ball mask for balls faced.

In [ ]:
# code here

# 6. Q-9 — Player-of-the-Match Figures

### Q-9: Using both datasets, make a dataframe as described below.

Columns: `['PlayerOfThematch', 'BattingFigure', 'BowlingFigure']`

* BattingFigure -> `<runs>/<balls>`
* BowlingFigure -> `<wicket>/<runs-conceded>`

The dataframe should have one record for each match.

Say 'V Kohli' got the POM award; then the dataset should include his batting figure in that match. If he scored 112 runs in 76 balls, and did not bowl, the Bowling Figure will be NaN.

```
PlayerOfThematch  BattingFigure  BowlingFigure
V Kohli           112/76         nan
```

This question needs BOTH files, and the merge key is the one you already know: `delivery.merge(matches, left_on='match_id', right_on='id')` joins every ball to its match.

The plan:

1. Merge, then locate the player of the match. For each match, the matches frame names the POM; the merged frame lets you filter `batsman == player_of_match` — but per match. The cleanest path is to group the merged balls by match_id, and within each match's group select the POM's rows.

2. BattingFigure per match: from the POM's rows in that match, `sum(batsman_runs)` over `count(balls faced)` — the string `f'{runs}/{balls}'`.

3. BowlingFigure: same match, same player, but now from their rows as `bowler`: `count(wickets)` over `sum(runs conceded)` — `f'{wickets}/{runs}'`. If the POM never bowled, the group has no rows with him as bowler, so the figure is NaN.

4. Assemble one row per match_id: PlayerOfThematch, BattingFigure, BowlingFigure. A natural approach is groupby('match_id').apply(helper_function) where the helper inspects the group's rows for the POM on both sides of the ball.

> Hint: merge on match_id/id first; group by match_id; inside each group pick the POM's batting rows and bowling rows; format the two figure strings.

In [ ]:
# code here

# 7. The Iris Datasets

The final project is the most famous flower in data science: Edgar Anderson's Iris measurements — 150 flowers, three species (Setosa, Versicolor, Virginica), four measurements per flower.

This task hands you the data deliberately SPLIT into five fragments:

- **sepal_all** — SepalLengthCm and SepalWidthCm for every flower.
- **petal_all** — PetalLengthCm and PetalWidthCm for every flower.
- **virginica**, **versicolor**, **setosa** — the species label for each of the three classes.

Each fragment carries an `Id` that identifies the flower, and an `Unnamed: 0` column that is an artefact of how the sub-datasets were exported — the definitions call it 'Unused column', so drop it.

```python
sepal_all  = pd.read_csv("https://docs.google.com/spreadsheets/...sepal.../pub?...&output=csv")
petal_all  = pd.read_csv("https://docs.google.com/spreadsheets/...petal.../pub?...&output=csv")
virginica  = pd.read_csv("https://docs.google.com/spreadsheets/...virginica.../pub?...&output=csv")
versicolor = pd.read_csv("https://docs.google.com/spreadsheets/...versicolor.../pub?...&output=csv")
setosa     = pd.read_csv("https://docs.google.com/spreadsheets/...setosa.../pub?...&output=csv")
```

Reading in the five URLs is one cell, and the data dictionary in the markdown above documents every column the rest of the project relies on: use `Id`, not the Unnamed artefact. From here, the three questions reconstruct the classic Iris table — plotting a comparison (Q-9), assembling and shuffling the full dataset (Q-10), and grouping by species for min/max widths (Q-11).

In [ ]:
import pandas as pd
sepal_all = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vT58ekmHTwptX7Bs4QOy6YByA1HMvYTACeeIjrKhHE0Pg1K_3egewHMKMh02zN9D5-yHVXfvuaa3s5u/pub?gid=2028782809&single=true&output=csv")
petal_all = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQinLXShrOz4ExNaW1bSQVuvbbhIzJW7G0kkkD2SvqSD6STjLrQQiftgI7BGe10sBZi0CNr2_sJpQAz/pub?gid=1580010789&single=true&output=csv")

virginica = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vSK39MwduGPHYNgw5yViezoLYCVDKMCWIHzjnt3GZNaxHPFOQLr2q6no_tyqTsOk-VfXleslfGVe9eJ/pub?gid=314231613&single=true&output=csv")
versicolor = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vTcSFgLnabqIrgIc5WlwvnbbvyyJsgZjR-0E0-4TR-5aHgv_0EP6yNWglkkls3AXM2qHCR5VYzWCoTM/pub?gid=715607857&single=true&output=csv")
setosa = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vSjqJpdgy2X_oDUUqQ0sSaFKqnnf8MYU4KgJSYgHaHmq0Wb1weMOsJXh-rICHmkLcaTkOwzMYLeh959/pub?gid=2003684803&single=true&output=csv")


# 8. Q-9 — Comparing Two Flower Species

### Q-9: Plot a bar chart of the average Sepal Length of Virginica and the average Petal Length of Setosa.

Two numbers, one bar chart. Every flower in the virginica fragment is Iris-virginica, so the average sepal length of Virginica is simply the mean of its sepal column: `virginica['SepalLengthCm'].mean()`. Likewise the average petal length of Setosa is `setosa['PetalLengthCm'].mean()`.

Build a two-value Series indexed by the categories:

```python
pd.Series({
    'Virginica Sepal Length': virginica['SepalLengthCm'].mean(),
    'Setosa Petal Length': setosa['PetalLengthCm'].mean(),
}).plot(kind='bar')
```

The bar chart shows the two averages side by side. The visual fact you will notice — and the reason the question picks these two — is that Virginica sepals are markedly longer than Setosa petals: Iris-virginica grows the longest sepals of the three species. Remember to give the Series meaningful labels so the chart reads by itself. If a title or rotated labels are wanted, `plt.title(...)` and `plt.xticks(rotation=45)` finish the figure; the `.plot(kind='bar')` defaults are already enough for the answer.

> Hint: two means placed in a labeled Series or DataFrame, then `.plot(kind='bar')`.

In [ ]:
# code here

# 9. Q-10 — Assembling the Full Dataset

### Q-10: Create the complete dataset by using the below datasets:
- virginica
- versicolor
- setosa
- sepal all
- petal all

The dataset should have these below column names in order:
1. Id
2. Species
3. SepalLengthCm
4. SepalWidthCm
5. PetalLengthCm
6. PetalWidthCm

Also, the dataset should be shuffled — the `Id` column should not be in increasing or decreasing order. So make a dataset which has the shuffled Id column. You can use `DataFrame.sample()` to shuffle.

The three species files each carry an `Id` and the species label; the two all-files carry `Id` alongside one measurement pair each. The reconstruction is merge- or concat-based:

1. Stack the species: `species = pd.concat([virginica, versicolor, setosa])` (the files match in structure).
2. Merge on Id for the measurements: first attach `sepal_all` (Id + SepalLengthCm + SepalWidthCm), then `petal_all` (Id + PetalLengthCm + PetalWidthCm), always `on='Id'`. Because every fragment covers the same 150 flowers, an inner merge on Id yields the full 150-row table with no NaN.
3. Column order: select exactly `['Id', 'Species', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']`.
4. Shuffle: `df.sample(frac=1)` returns all rows in random order — the Id column no longer marches 1, 2, 3, ..., 150. Optionally `reset_index(drop=True)` for a clean index.

The result is the familiar Iris dataset ready for plotting or modelling — exactly how the original CSV looked before the fragments were created.

> Hint: concat species -> merge sepal_all, then petal_all, on Id -> select the six columns -> df.sample(frac=1).

In [ ]:
# code here

# 10. Q-11 — Width Extremes by Species

### Q-11: Find out the maximum and minimum sepal width and petal width of Setosa and Versicolor. To do this:
- First create a dataset by merging the required datasets.
- After that, use `groupby` to create groups based on the Species column.
- Then find out which are asked in this question.

The output should be like:
```bash
Minimum Sepal width of Setosa is 2.3
Maximum Sepal width of Setosa is 4.4

**************************************************

Minimum Sepal width of Versicolor is 2.0
Maximum Sepal width of Versicolor is 3.4
...
```

Reuse the assembled frame from Q-10 (or rebuild it: merge the species with the two all-files on Id). Then the machinery is pure GroupBy: `full.groupby('Species')` creates three species groups, and the width columns answer the question with min and max.

`full.groupby('Species')[['SepalWidthCm', 'PetalWidthCm']].agg(['min', 'max'])` returns one table with every request — the minimum and maximum sepal width and petal width per species in a single read.

The OUTPUT FORMAT is the real requirement: the question prints natural-language sentences separated by a line of asterisks. So the answer is a short loop over the wanted species and attributes, pulling each value from the groupby result and printing with an f-string, with the asterisk separator between blocks. Verify the numbers against the sample: Setosa's sepal width spans 2.3 to 4.4, Versicolor's 2.0 to 3.4. The habit to take away: aggregate once with groupby, then format text for output — not the other way around.

> Hint: groupby('Species'), agg min/max on the widths, then an f-string loop to print the sentences.

## Summary

Task 20 closed the pandas arc with three full mini-projects. On the store data you stacked three quarterly order files into one frame with keys as a Q-1/Q-2/Q-3 index, merged in the item catalogue, converted dollar-string prices to rupees, and answered the per-quarter questions: most-sold items, highest-revenue items, and average order price. On the IPL ball-by-ball log you computed the purple-cap holder per season and the best death-over bowler using wickets with economy as the tiebreak, wrote a per-batsman season-by-season record function covering innings, runs, average, highest score, and a wide-aware strike rate, and built a per-match player-of-the-match report of batting and bowling figures by merging matches with deliveries. On the iris fragments you plotted average Virginica sepal length against Setosa petal length, reassembled the five files into the full six-column dataset and shuffled it with sample(frac=1), and used groupby by Species to print the min and max widths for Setosa and Versicolor. Every question was a chain of read, merge, group, and filter.

In [ ]:
# code here